# Kaggle Tabular: Airline Passenger Satisfaction
Colab-ready pipeline for Exploratory Data Analysis, Stratified K-Fold CV, LightGBM modeling, and submission generation.

In [ ]:
# 1. Setup & Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42

In [ ]:
# 2. Locate and Load Datasets
# Checks data/raw, data, or current working directory
search_dirs = [Path('data/raw'), Path('data'), Path('.')]
train_path, test_path = None, None

for d in search_dirs:
    if d.exists():
        for p in d.glob('*.csv'):
            if 'train' in p.name.lower() and not train_path:
                train_path = p
            elif 'test' in p.name.lower() and not test_path:
                test_path = p

print(f'Train path: {train_path}')
print(f'Test path:  {test_path}')

train_df = pd.read_csv(train_path) if train_path else pd.DataFrame()
test_df = pd.read_csv(test_path) if test_path else pd.DataFrame()

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')

In [ ]:
# 3. Quick Overview & Target Identification
if not train_df.empty:
    display(train_df.head())
    display(train_df.info())
    target_cols = [c for c in train_df.columns if c not in test_df.columns]
    print(f'Target candidate column(s): {target_cols}')

In [ ]:
# 4. Preprocessing & Feature Encoding
def prepare_features(train: pd.DataFrame, test: pd.DataFrame):
    tr = train.copy()
    te = test.copy()

    id_cols = [c for c in ['id', 'ID', 'Unnamed: 0'] if c in te.columns]
    test_ids = te[id_cols[0]] if id_cols else pd.Series(te.index, name='id')

    target_col = [c for c in tr.columns if c not in te.columns][0]
    y = tr[target_col]

    # Map string target if binary satisfaction
    if y.dtype == object:
        y = y.map(lambda v: 1 if 'satisfied' in str(v).lower() and 'neutral' not in str(v).lower() else 0)

    X = tr.drop(columns=[target_col] + id_cols, errors='ignore')
    X_test = te.drop(columns=id_cols, errors='ignore')

    # Categorical cast for tree algorithms
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].astype('category')
        X_test[col] = X_test[col].astype('category')

    return X, y, X_test, test_ids, target_col

if not train_df.empty and not test_df.empty:
    X, y, X_test, test_ids, target_name = prepare_features(train_df, test_df)
    print(f'Processed X shape: {X.shape}, y distribution:
{y.value_counts(normalize=True)}')

In [ ]:
# 5. 5-Fold Stratified LightGBM Baseline
if not train_df.empty and not test_df.empty:
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        clf = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.05,
            random_state=RANDOM_SEED + fold,
            n_jobs=-1
        )

        clf.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        val_pred = clf.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_pred
        test_preds += clf.predict_proba(X_test)[:, 1] / n_splits

        auc = roc_auc_score(y_val, val_pred)
        print(f'Fold {fold + 1} ROC-AUC: {auc:.5f}')

    overall_auc = roc_auc_score(y, oof_preds)
    print(f'\nOverall Out-of-Fold ROC-AUC: {overall_auc:.5f}')

In [ ]:
# 6. Generate Submission CSV
if not test_df.empty:
    os.makedirs('submissions', exist_ok=True)
    sub = pd.DataFrame()
    sub[test_ids.name or 'id'] = test_ids
    # Default binary output threshold 0.5 (or keep test_preds if metric is AUC/prob)
    sub[target_name] = (test_preds >= 0.5).astype(int)
    
    sub_path = 'submissions/submission_lgbm_baseline.csv'
    sub.to_csv(sub_path, index=False)
    print(f'Saved: {sub_path}')
    display(sub.head())